# **Generate Masks**

This notebook processes long `.tif` time series by segmenting each frame individually using a pretrained `Cellpose` model and tracking masks across frames to ensure consistent cell labels over time.

### **Import Libraries**

We begin by importing the necessary packages, including:
- `Cellpose` model API
- `tifffile` for reading/writing `.tif` stacks
- `numpy` and `tqdm` for image and mask manipulation
- `cct_utils`, which contains our custom tracking logic

In [ ]:
import os
import tifffile
import numpy as np
from tqdm import tqdm
from cellpose import models
cct_utils = __import__('0_cct_utils')

### **Define Paths**

Here we define the relevant paths:
- The `.tif` file to process
- The location of the pretrained `Cellpose` model
- The directory for saving masks

In [ ]:
model_name = "cellpose_1746802277.9571602"
model_folder = "ModelAB1"
file_name = 'CON_cluster1'

parent_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
raw_path = os.path.normpath(os.path.join(parent_dir, "raw_data"))
mask_path = os.path.normpath(os.path.join(parent_dir, "masks_tracked", model_folder))
model_path = os.path.normpath(os.path.join(parent_dir, "saved_models", model_folder, "cellpose_train", "models", model_name))

raw_file = os.path.join(raw_path, f"{file_name}.tif")
output_name = f"{file_name}_mask.tif"

#TODO: evaluate the effect of different settings for diameter and dist_limit 
#TODO: evaluate the effect of changing flow_threshold
#TODO: continue looking at the tracking function to make it avoid putting the same label on several cells

diam = 30 # Standard setting when trained
dist_limit = diam/2 #Max pixel distance a cell’s COM can move between frames to be counted as the same cell; increase if cells move a lot.
backtrack_limit = 5 #Max number of images back that the algorithm will search through to find a center of mass within the distance limit

os.makedirs(mask_path, exist_ok=True)

### **Tracking Function**

The `track_Y` function:
- Segments each frame individually using the pretrained `Cellpose` model
- Tracks cells across frames using custom tracking logic from `cct_utils`

In [ ]:
def track_Y(X, model, diam=diam, dist_limit=dist_limit, backtrack_limit=backtrack_limit, save=False, name=None, savedir=None):
    crop_idx = np.argmax(np.mean(X, axis=(1, 2)) == 0)
    X = X[:crop_idx] if crop_idx > 0 else X

    Y = [model.eval(np.squeeze(i),
                    diameter=diam,
                    channels=[0, 0],
                    flow_threshold=2.0,
                    cellprob_threshold=0,
                    do_3D=False)[0] for i in tqdm(np.split(X, X.shape[0]), desc='Segmenting timeseries', unit=' frames')]
    
    print('\nTracking initiated...')
    tracked_Y = cct_utils.get_tracked_masks(masks=np.array(Y), dist_limit=dist_limit, backtrack_limit=backtrack_limit, save=save, name=name, savedir=savedir)
    return tracked_Y

### **Load, Segment, Track, and Save**

- Load the full `.tif` image stack
- Initialize a pretrained `Cellpose` model for segmentation
- Process the stack sequentially, segmenting and tracking cells across frames
- Generate a final 3D mask stack with consistent cell labels
- Save the final mask stack as a `.tif` file in the designated output directory

In [ ]:
X = tifffile.imread(raw_file)
model = models.CellposeModel(gpu=True, pretrained_model=model_path)
print("Processing entire stack frame-by-frame...")
final_masks = track_Y(X, model, diam=None, dist_limit=dist_limit, backtrack_limit=backtrack_limit, save=True, name=output_name, savedir=mask_path)

### **Optional: Save Key Raw Frames**
- Saves key frames corresponding to the start, middle, and end of the raw time-series in the list `file_names`.

In [ ]:
# List of TIFF files (without extension)
file_names = [
    'CON_cluster1',
    'CON_cluster2',
    'OUA_cluster1',
    'OUA_cluster2'
]

output_path = os.path.normpath(os.path.join(parent_dir, "plots", "raw_plots"))

# Loop through all files
for file_name in file_names:
    raw_file = os.path.join(raw_path, f"{file_name}.tif")
    if os.path.exists(raw_file):
        cct_utils.save_raw_frames_as_pdfs(file_name, raw_file, output_path)
    else:
        print(f"Warning: File not found - {raw_file}")